# Data Cleaning Notebook
This notebook performs step-by-step cleaning of the raw data files.

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
import logging

# Set data path
DATA_DIR = Path('../data')
logging.basicConfig(level=logging.WARNING, format='%(levelname)s: %(message)s')

## 1. Load Customers Data

In [4]:
cust_df = pd.read_csv(DATA_DIR / 'customers.csv')
print(f"Initial rows: {len(cust_df)}")
cust_df.head()

Initial rows: 10


,customer_id,name,email,region,signup_date
0,C001,Alice Johnson,alice@example.com,North,2023-01-15
1,C002,Bob Smith,bob@test.com,South,2023-02-20
2,C003,Charlie Brown,NaN,East,2023-03-10
3,C004,David Wilson,david@example.com,West,2023-04-05
4,C005,Eva Garcia,NaN,Central,2023-05-12


## 1.1 Remove Duplicates
Remove duplicate rows based on `customer_id`, keeping the most recent `signup_date`.

In [5]:
# First parse dates temporarily to sort correctly
cust_df['signup_date'] = pd.to_datetime(cust_df['signup_date'], errors='coerce')
cust_df = cust_df.sort_values(by=['customer_id', 'signup_date'], ascending=[True, False])

# Drop duplicates
before_count = len(cust_df)
cust_df = cust_df.drop_duplicates(subset='customer_id', keep='first')
print(f"Rows removed: {before_count - len(cust_df)}")
cust_df.head()

Rows removed: 2


,customer_id,name,email,region,signup_date
0,C001,Alice Johnson,alice@example.com,North,2023-01-15
1,C002,Bob Smith,bob@test.com,South,2023-02-20
2,C003,Charlie Brown,NaN,East,2023-03-10
3,C004,David Wilson,david@example.com,West,2023-04-05
4,C005,Eva Garcia,NaN,Central,2023-05-12


## 1.2 Standardize Emails
Standardize email addresses to lowercase and flag missing or malformed emails.

In [6]:
cust_df['email'] = cust_df['email'].astype(str).str.lower().str.strip()
cust_df.loc[cust_df['email'].isin(['nan', 'none', '']), 'email'] = ''

# Validation flag
cust_df['is_valid_email'] = cust_df['email'].apply(lambda x: isinstance(x, str) and '@' in x and '.' in x if x else False)
cust_df[['customer_id', 'email', 'is_valid_email']].head()

,customer_id,email,is_valid_email
0,C001,alice@example.com,True
1,C002,bob@test.com,True
2,C003,NaN,False
3,C004,david@example.com,True
4,C005,NaN,False


## 1.3 Parse Signup Date
Parse signup_date into YYYY-MM-DD and log warnings for unparseable values.

In [7]:
cust_df['signup_date'] = pd.to_datetime(cust_df['signup_date'], errors='coerce')
unparseable = cust_df['signup_date'].isna().sum()
if unparseable > 0:
    print(f"WARNING: Found {unparseable} unparseable signup_date values.")

cust_df['signup_date'] = cust_df['signup_date'].dt.strftime('%Y-%m-%d')
cust_df.head()

,customer_id,name,email,region,signup_date,is_valid_email
0,C001,Alice Johnson,alice@example.com,North,2023-01-15,True
1,C002,Bob Smith,bob@test.com,South,2023-02-20,True
2,C003,Charlie Brown,NaN,East,2023-03-10,False
3,C004,David Wilson,david@example.com,West,2023-04-05,True
4,C005,Eva Garcia,NaN,Central,2023-05-12,False


## 1.4 Strip Whitespace
Strip leading/trailing whitespace from `name` and `region` columns.

In [8]:
cust_df['name'] = cust_df['name'].astype(str).str.strip()
cust_df['region'] = cust_df['region'].astype(str).str.strip()
cust_df.head()

,customer_id,name,email,region,signup_date,is_valid_email
0,C001,Alice Johnson,alice@example.com,North,2023-01-15,True
1,C002,Bob Smith,bob@test.com,South,2023-02-20,True
2,C003,Charlie Brown,NaN,East,2023-03-10,False
3,C004,David Wilson,david@example.com,West,2023-04-05,True
4,C005,Eva Garcia,NaN,Central,2023-05-12,False


## 1.5 Fill Missing Regions
Fill missing region with the string 'Unknown'.

In [9]:
cust_df['region'] = cust_df['region'].replace(['None', 'nan', ''], 'Unknown')
print("Value counts for region:")
print(cust_df['region'].value_counts())
cust_df.head()

Value counts for region:
region
North      2
South      2
East       2
West       1
Central    1
Name: count, dtype: int64


,customer_id,name,email,region,signup_date,is_valid_email
0,C001,Alice Johnson,alice@example.com,North,2023-01-15,True
1,C002,Bob Smith,bob@test.com,South,2023-02-20,True
2,C003,Charlie Brown,NaN,East,2023-03-10,False
3,C004,David Wilson,david@example.com,West,2023-04-05,True
4,C005,Eva Garcia,NaN,Central,2023-05-12,False


## 2. Save Cleaned Customers

In [10]:
cust_df.to_csv(DATA_DIR / 'customers_clean.csv', index=False)
print("Successfully saved customers_clean.csv")

Successfully saved customers_clean.csv
